# Data cleaning <br>

Load the data. Only data after 1980 is included for quick code execution.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

engine = create_engine(
    "mysql+pymysql://root@localhost:3306/climate_watch"
)

df_city = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCity WHERE dt > '1980-01-01'", con=engine, parse_dates=["dt"])
df_country = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCountry WHERE dt > '1980-01-01'", con=engine, parse_dates=["dt"])
df_major_city = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByMajorCity WHERE dt > '1980-01-01'", con=engine, parse_dates=["dt"])
df_state = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByState WHERE dt > '1980-01-01'", con=engine, parse_dates=["dt"])
df_global = pd.read_sql("SELECT * FROM GlobalTemperatures WHERE dt > '1980-01-01'", con=engine, parse_dates=["dt"])

## Often used functions

In [2]:
def fill_na_with_avg(df: pd.DataFrame, fill_column: str, group_column: list[str], sort_columns: list[str]) -> pd.DataFrame:
    """
    First, sort the dataframe by 'sort_columns'.
    Then replace the missing values of 'fill_column' by the mean value of the previous en next value of that 'group_column'.
    Note that this will not work if there are 3 consecutive missing values.
    """
    df = df.sort_values(sort_columns)
    df["previous"] = df.groupby(group_column)[fill_column].shift(1)
    df["next"] = df.groupby(group_column)[fill_column].shift(-1)
    df["average"] = df[["previous", "next"]].mean(skipna=True, axis=1)
    df[fill_column] = df[fill_column].fillna(df["average"])
    df = df.drop(columns=["previous", "next", "average"])

    return df

def update_data_in_db(df: pd.DataFrame, table_name: str) -> None:
    """
    To keep the table structure, I will first delete all data and then append the current data to the table.
    """
    with engine.begin() as connection:
        connection.execute(text(f"DELETE FROM {table_name}"))
        df.to_sql(
            name=table_name,
            con=connection,
            if_exists="append",
            index=False,
            chunksize=5000
            )

## City data

In [3]:
for col in df_city.columns:
    print(f"{col}: {"Missing values" if df_city[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per city:")
df_city[df_city.isna().any(axis=1)].City.value_counts()

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
City: No missing values
Country: No missing values
Latitude: No missing values
Longitude: No missing values


Nr of missing values per city:


City
Rongcheng    3
Barcelona    2
Bharatpur    2
Brest        2
Cadiz        2
            ..
Zouxian      1
Zunyi        1
Zurich       1
Zuwarah      1
Zwolle       1
Name: count, Length: 3045, dtype: int64

There are few options for dealing with the missing values:
- Remove any row with missing values
- Replace missing values by 0
- Replace missing value by avg value of that country
- Replace missing value by avg value of that country in the same month and same year
- Replace missing value by avg of previous en next value of that country

I will go with the last option, as this should give a gradual progression of the temperature.

In [4]:
df_city = fill_na_with_avg(df_city, "AverageTemperature", ["City", "Country"], ["City", "Country", "dt"])
df_city = fill_na_with_avg(df_city, "AverageTemperatureUncertainty", ["City", "Country"], ["City", "Country", "dt"])

update_data_in_db(df_city, "GlobalLandTemperaturesByCity")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18840\1045732381.py:22: UserWarning: The provided table name 'GlobalLandTemperaturesByCity' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## Country data

In [5]:
for col in df_country.columns:
    print(f"{col}: {"Missing values" if df_country[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per country:")
display(df_country[df_country.isna().any(axis=1)].Country.value_counts())
print(f"{len(df_country[df_country["Country"] == "Antarctica"])} rows from Antarctica are missing data.")

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
Country: No missing values


Nr of missing values per country:


Country
Antarctica        404
Åland               1
Afghanistan         1
Africa              1
Albania             1
                 ... 
Vietnam             1
Western Sahara      1
Yemen               1
Zambia              1
Zimbabwe            1
Name: count, Length: 222, dtype: int64

404 rows from Antarctica are missing data.


In [6]:
df_country = df_country[df_country["Country"] != "Antarctica"].reset_index(drop=True)
df_country = fill_na_with_avg(df_country, "AverageTemperature", ["Country"], ["Country", "dt"])
df_country = fill_na_with_avg(df_country, "AverageTemperatureUncertainty", ["Country"], ["Country", "dt"])

update_data_in_db(df_country, "GlobalLandTemperaturesByCountry")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18840\1045732381.py:22: UserWarning: The provided table name 'GlobalLandTemperaturesByCountry' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## Major city data

In [7]:
for col in df_major_city.columns:
    print(f"{col}: {"Missing values" if df_major_city[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per major city:")
df_major_city[df_major_city.isna().any(axis=1)].City.value_counts()

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
City: No missing values
Country: No missing values
Latitude: No missing values
Longitude: No missing values


Nr of missing values per major city:


City
Abidjan        1
Addis Abeba    1
Ahmadabad      1
Aleppo         1
Alexandria     1
              ..
Tianjin        1
Tokyo          1
Umm Durman     1
Wuhan          1
Xian           1
Name: count, Length: 93, dtype: int64

In [8]:
df_major_city = fill_na_with_avg(df_major_city, "AverageTemperature", ["City", "Country"], ["City", "Country", "dt"])
df_major_city = fill_na_with_avg(df_major_city, "AverageTemperatureUncertainty", ["City", "Country"], ["City", "Country", "dt"])

update_data_in_db(df_major_city, "GlobalLandTemperaturesByMajorCity")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18840\1045732381.py:22: UserWarning: The provided table name 'GlobalLandTemperaturesByMajorCity' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## State data

In [9]:
for col in df_state.columns:
    print(f"{col}: {"Missing values" if df_state[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per state:")
df_state[df_state.isna().any(axis=1)].State.value_counts()

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
State: No missing values
Country: No missing values


Nr of missing values per state:


State
Acre            1
Adygey          1
Aga Buryat      1
Alagoas         1
Alaska          1
               ..
Yamal Nenets    1
Yaroslavl'      1
Yevrey          1
Yunnan          1
Zhejiang        1
Name: count, Length: 181, dtype: int64

In [10]:
df_state = fill_na_with_avg(df_state, "AverageTemperature", ["State", "Country"], ["State", "Country", "dt"])
df_state = fill_na_with_avg(df_state, "AverageTemperatureUncertainty", ["State", "Country"], ["State", "Country", "dt"])

update_data_in_db(df_state, "GlobalLandTemperaturesByState")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18840\1045732381.py:22: UserWarning: The provided table name 'GlobalLandTemperaturesByState' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## Global data

In [11]:
for col in df_global.columns:
    print(f"{col}: {"Missing values" if df_global[col].isna().any() else "No missing values"}")

dt: No missing values
LandAverageTemperature: No missing values
LandAverageTemperatureUncertainty: No missing values
LandMaxTemperature: No missing values
LandMaxTemperatureUncertainty: No missing values
LandMinTemperature: No missing values
LandMinTemperatureUncertainty: No missing values
LandAndOceanAverageTemperature: No missing values
LandAndOceanAverageTemperatureUncertainty: No missing values
